In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import os
import sys
sys.path.insert(0, "/".join(["..", "python"]))

import matplotlib.pyplot as plt
import numpy as np
from gridr.core.convolution.fft_filtering import fft_array_filter
from gridr.misc.mandrill import mandrill
from notebook_utils import plot_im, mpl_plot_wrapper
IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
import notebook_utils
notebook_utils.FORCE_MPL_INTERACTIVE = True
if not IN_DOC_BUILD and not notebook_utils.FORCE_MPL_INTERACTIVE:
    from bokeh.io import output_notebook  # enables plot interface in J notebook
    output_notebook()


def gaussian_kernel_2d(shape, sigma):
    """
    Generates a normalized 2D Gaussian kernel.

    :param shape: Kernel size (int for a square kernel, or tuple (height, width))
    :param sigma: Standard deviation of the Gaussian (float or tuple (sigma_y, sigma_x))
    :return: a tuple containing:
        - 1D Numpy array representing x coordinates
        - 1D Numpy array representing y coordinates
        - 2D NumPy array representing the kernel
    """
    # Handle scalar or tuple inputs for shape and sigma
    if isinstance(shape, int):
        h = w = shape
    else:
        h, w = shape

    if isinstance(sigma, (int, float)):
        sigma_y = sigma_x = sigma
    else:
        sigma_y, sigma_x = sigma

    # Create centered coordinates (e.g., size 5 -> [-2, -1, 0, 1, 2])
    y = np.arange(-(h // 2), h // 2 + 1 if h % 2 != 0 else h // 2)
    x = np.arange(-(w // 2), w // 2 + 1 if w % 2 != 0 else w // 2)

    # 2D Grid
    x_grid, y_grid = np.meshgrid(x, y)

    # Calculate 2D Gaussian function
    kernel = np.exp(-0.5 * ((x_grid / sigma_x) ** 2 + (y_grid / sigma_y) ** 2))

    # Normalize so the sum of all elements equals 1
    kernel_sum = np.sum(kernel)
    if kernel_sum != 0:
        kernel /= kernel_sum

    return x, y, kernel


@mpl_plot_wrapper
def plot_mesh3d_static(x, y, data, *, height=350, width=350, prefix=None):
    """Plot a 2D array as a 3D wireframe surface."""
    fig = plt.figure(figsize=(width / 100, height / 100), dpi=100)
    ax = fig.add_subplot(111, projection="3d")
    X, Y = np.meshgrid(x, y)
    ax.plot_wireframe(X, Y, data, rstride=1, cstride=1, color="blue", linewidth=0.3)
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    return fig


mandrill_0 = mandrill[0]

# The `gaussian_kernel_2d` function implements the formula given in the first tutorial.
gaussian_sigma = 4.
x, y, kernel = gaussian_kernel_2d(shape=int(gaussian_sigma * 6 + 1), sigma=gaussian_sigma)

# Getting started: your first filtering

This first tutorial presents the simplest call to ``fft_array_filter``, which computes the
FFT-based convolution of an image with a user-defined kernel.

**What you'll learn**

- How to import and call ``fft_array_filter``
- How to define a normalized Gaussian kernel from its standard deviations
- What the ``FilterResult`` return type contains

## Setting things up

We first import `fft_array_filter`, together with the well known mandrill raster
image used as test data throughout this guide.

In [ ]:
from gridr.core.convolution.fft_filtering import fft_array_filter
from gridr.misc.mandrill import mandrill

The mandrill image has three colour channels. We keep the first one, so that the
examples operate on a single 2D array.

In [ ]:
mandrill_0 = mandrill[0]

The last element to define is the convolution kernel. We use a discrete normalized
Gaussian of size $(2r + 1) \times (2r + 1)$, given by:

$$
K(i, j) = \frac{1}{Z}
\exp\left( -\frac{1}{2} \left( \frac{i^2}{\sigma_y^2} + \frac{j^2}{\sigma_x^2} \right) \right),
\qquad -r \leq i, j \leq r
$$

where $i$ and $j$ are the row and column offsets from the kernel center, and $Z$ is the sum
of the unnormalized values, so that the kernel sums to 1.

In this example we set $\sigma_x = \sigma_y = 4$ and $r = 3\sigma = 12$, which gives a
25x25 kernel covering the Gaussian up to three standard deviations from its center.

In [ ]:
# `gaussian_kernel_2d` implements the formula above.
gaussian_sigma = 4.
x, y, kernel = gaussian_kernel_2d(shape=int(gaussian_sigma * 6 + 1),
                                  sigma=gaussian_sigma)

The kernel is shown below as a surface.

In [ ]:
_ = plot_mesh3d_static(x, y, kernel, prefix="001_gaussian_kernel_sigma4")

## Filtering the image

We call `fft_array_filter` with only the required parameters and keep the default
values for the others. These optional parameters are described later in this guide.

In [ ]:
filtered_mandrill_0 = fft_array_filter(
        arr=mandrill_0,
        kernel=kernel,
    )

`fft_array_filter` returns a `FilterResult`, which is a `NamedTuple` with two fields.
The `data` field contains the filtered image. The `window` field gives the position of
the production window (the whole input image by default) in the coordinates of the full
output. This window is discussed in the next tutorial.

The type of the result, then the type and shape of the filtered array:

In [ ]:
type(filtered_mandrill_0)

In [ ]:
type(filtered_mandrill_0.data), filtered_mandrill_0.data.shape

Input and output can now be compared side by side.

In [ ]:
plot_im({"input": mandrill_0, "output": filtered_mandrill_0.data},
        prefix="001_filtered_mandrill_0")

## What's next

You have now blurred your first image with GridR. The edges of the filtered image appear
darker than the rest: the convolution needs values beyond the image borders, and none are
available there. The next tutorial, *Boundary conditions and output modes*, explains how
these values are produced and how to choose which part of the result is returned.